In [14]:
from mlx_vlm import load, apply_chat_template, generate
from mlx_vlm.utils import load_image

In [15]:
deepseek_ocr_model, deepseek_ocr_processor = load("mlx-community/DeepSeek-OCR-8bit")
deepseek_ocr_config = deepseek_ocr_model.config

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 85598.04it/s]


Add pad token = ['<｜▁pad▁｜>'] to the tokenizer
<｜▁pad▁｜>:2
Add image token = ['<image>'] to the tokenizer
<image>:128815
Added grounding-related tokens
Added chat tokens


In [16]:
type(deepseek_ocr_model)

mlx_vlm.models.deepseekocr.deepseekocr.Model

In [17]:
import os
import fitz
import img2pdf
import io
import re
from tqdm import tqdm
import sys
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from io import BytesIO
import numpy as np
import base64


In [18]:
MODEL_CONFIGS = {
    "Gundam": {"base_size": 1024, "image_size": 640, "crop_mode": True},
    "Tiny": {"base_size": 512, "image_size": 512, "crop_mode": False},
    "Small": {"base_size": 640, "image_size": 640, "crop_mode": False},
    "Base": {"base_size": 1024, "image_size": 1024, "crop_mode": False},
    "Large": {"base_size": 1280, "image_size": 1280, "crop_mode": False}
}

TASK_PROMPTS = {
    "Markdown": {"prompt": "<|grounding|>Convert the document to markdown.", "has_grounding": True},
    "Free OCR": {"prompt": "<image>\nFree OCR.", "has_grounding": False},
    "Locate": {"prompt": "<image>\nLocate <|ref|>text<|/ref|> in the image.", "has_grounding": True},
    "Describe": {"prompt": "<image>\nDescribe this image in detail.", "has_grounding": False},
    "Custom": {"prompt": "", "has_grounding": False}
}
INPUT_PATH = "/Users/jajajou1778/UIT_DOCS_AGENT/firecrawl/data/daa/quydinh_huongdan/huong-dan-chuan-qua-trinh/pdf/560-qd-dhcntt_5-6-2024_sua_doi_quy_dinh_dao_tao_ngoai_ngu_doi_voi_he_dai_hoc_chinh_quy.pdf"
OUTPUT_DIR = "/Users/jajajou1778/UIT_DOCS_AGENT/data/deepseek_ocr_results"
SKIP_REPEAT = True



In [19]:
def draw_bounding_boxes(image, refs, extract_images=False):

    img_w, img_h = image.size
    img_draw = image.copy()
    draw = ImageDraw.Draw(img_draw)

    overlay = Image.new('RGBA', img_draw.size, (0, 0, 0, 0))
    draw2 = ImageDraw.Draw(overlay)
    crops = []

    
    #     except IOError:
    font = ImageFont.load_default()
    np.random.seed(42)


    color_map = {}
    
    for ref in refs:
        label = ref[1]
        if label not in color_map:
            color_map[label] = (np.random.randint(50, 255), np.random.randint(50, 255), np.random.randint(50, 255))

        color = color_map[label]
        coords = eval(ref[2])
        color_a = color + (60,)
        
        for box in coords:
            x1, y1, x2, y2 = int(box[0]/999*img_w), int(box[1]/999*img_h), int(box[2]/999*img_w), int(box[3]/999*img_h)
            
            if extract_images and label == 'image':
                crops.append(image.crop((x1, y1, x2, y2)))
            
            width = 5 if label == 'title' else 3
            draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
            draw2.rectangle([x1, y1, x2, y2], fill=color_a)
            
            text_bbox = draw.textbbox((0, 0), label, font=font)
            tw, th = text_bbox[2] - text_bbox[0], text_bbox[3] - text_bbox[1]
            ty = max(0, y1 - 20)
            draw.rectangle([x1, ty, x1 + tw + 4, ty + th + 4], fill=color)
            draw.text((x1 + 2, ty + 2), label, font=font, fill=(255, 255, 255))
    
    img_draw.paste(overlay, (0, 0), overlay)
    return img_draw, crops


In [20]:
def extract_grounding_references(text):
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    return re.findall(pattern, text, re.DOTALL)

def embed_images(markdown, crops):
    if not crops:
        return markdown
    for i, img in enumerate(crops):
        buf = BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        markdown = markdown.replace(f'**[Figure {i + 1}]**', f'\n\n![Figure {i + 1}](data:image/png;base64,{b64})\n\n', 1)
    return markdown

def clean_output(text, include_images=False, remove_labels=False):
    if not text:
        return ""
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)
    img_num = 0
    
    for match in matches:
        ref_type = match[1]
        ref_content = match[1]
        
        if ref_type == 'image':
            if include_images:
                text = text.replace(match[0], f'\n\n**[Figure {img_num + 1}]**\n\n', 1)
                img_num += 1
            else:
                text = text.replace(match[0], '', 1)
        elif ref_type == 'title':
            # Keep title tags for markdown formatting
            if not remove_labels:
                text = text.replace(match[0], '', 1)  # Remove the tags but keep the content
            else:
                text = text.replace(match[0], '', 1)
        elif ref_type == 'table':
            # Tables are already in markdown format, just remove the tags
            text = text.replace(match[0], '', 1)
        else:
            # For text and other types
            if remove_labels:
                text = text.replace(match[0], '', 1)
            else:
                text = text.replace(match[0], '', 1)  # Remove tags, keep content
    
    return text


In [21]:
def ocr_page(image: Image.Image, prompt: str = "Convert the document to markdown."):
    
    # temp_path = "temp_page.png"
    # image.save(temp_path)
    
    prompt = TASK_PROMPTS["Markdown"]["prompt"]
    has_grounding = TASK_PROMPTS["Markdown"]["has_grounding"]

    # temp_img = load_image(temp_path)
    # temp_img.resize(
    #     (1024, 1024)
    # )

    messages = [
        {"role": "user", "content": f"{prompt}"}
    ]

    formatted = apply_chat_template(
        deepseek_ocr_processor,
        deepseek_ocr_config,
        messages,
        num_images=1
    )

    output = generate(
        model=deepseek_ocr_model,
        processor=deepseek_ocr_processor, # type: ignore
        prompt=formatted, # type: ignore
        image=[image.resize((1024, 1024))], # type: ignore
        temperature=0.0,
        max_tokens=2048,
        verbose=True
    )

    # Extract just the text from GenerationResult
    if hasattr(output, 'text'):
        output = output.text
    else:
        output = str(output)
        # If it's still a string representation of GenerationResult, extract the text part
        if 'GenerationResult(text=' in output:
            import re
            match = re.search(r"text='(.*?)',\s*token=", output, re.DOTALL)
            if match:
                output = match.group(1)
                # Unescape any escaped characters
                output = output.replace("\\'", "'").replace("\\n", "\n").replace("\\\\", "\\")
    
    # Clean up the output - remove the verbose logging lines
    output = '\n'.join([l for l in str(output).split('\n') 
                        if not any(s in l for s in ['image:', 'other:', 'PATCHES', '====', 'BASE:', '%|', 'torch.Size'])]).strip()

    if not output:
        return "No text", "", "", None, []

    cleaned = clean_output(output, True, False)
    markdown = clean_output(output, False, False)

    img_out = None
    crops = []
    
    if has_grounding and '<|ref|>' in output:
        refs = extract_grounding_references(output)
        if refs:
            img_out, crops = draw_bounding_boxes(image, refs, True)
    
    # markdown = embed_images(markdown, crops)
    
    return cleaned, markdown, output, img_out, crops

In [22]:
def process_pdf(path):
    doc = fitz.open(path)
    image_format="PNG"
    texts, markdowns, raws, all_crops = [], [], [], []
    
    for i in range(len(doc)):
        print(f"page: {i+1} \n")
        page = doc.load_page(i)
        pix = page.get_pixmap(matrix=fitz.Matrix(144/72, 144/72), alpha=False)
        Image.MAX_IMAGE_PIXELS = None

        if image_format.upper() == "PNG":
            img_data = pix.tobytes("png")
            img = Image.open(io.BytesIO(img_data))
        else:
            img_data = pix.tobytes("png")
            img = Image.open(io.BytesIO(img_data))
            if img.mode in ('RGBA', 'LA'):
                background = Image.new('RGB', img.size, (255, 255, 255))
                background.paste(img, mask=img.split()[-1] if img.mode == 'RGBA' else None)
                img = background
        text, md, raw, _, crops = ocr_page(img)

        if text and text != "No text":
            texts.append(f"{text}")
            markdowns.append(md)
            raws.append(f"{raw}")
            all_crops.extend(crops)
    
    doc.close()
    
    return ("\n\n---\n\n".join(texts) if texts else "No text in PDF",
            "".join(markdowns) if markdowns else "No text in PDF",
            "\n\n".join(raws), None, all_crops)

In [23]:
text, markdown, raw, img_out, crops = process_pdf(INPUT_PATH)

page: 1 

Files: [<PIL.Image.Image image mode=RGB size=1024x1024 at 0x11DC8D0D0>] 

Prompt: <image>
<|grounding|>Convert the document to markdown. 
BASE:  (1, 256, 1280)
PATCHES:  (4, 100, 1280)
<|ref|>title<|/ref|><|det|>[[123, 68, 410, 128]]<|/det|>
# ĐẠI HỌC QUỐC GIA TP.HCM TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN 

<|ref|>text<|/ref|><|det|>[[162, 142, 365, 163]]<|/det|>
Số: 560/QĐ-ĐHCNTT 

<|ref|>text<|/ref|><|det|>[[500, 143, 880, 164]]<|/det|>
Tp.Hồ Chí Minh, ngày lýtháng 6 năm 2024 

<|ref|>sub_title<|/ref|><|det|>[[440, 180, 604, 202]]<|/det|>
## QUYẾT ĐỊNH 

<|ref|>text<|/ref|><|det|>[[211, 201, 870, 241]]<|/det|>
Về việc sửa đổi Quy định đào tạo ngoại ngữ đối với hệ đại học chính quy của Trường Đại học Công nghệ Thông tin 

<|ref|>sub_title<|/ref|><|det|>[[211, 260, 844, 283]]<|/det|>
## HIỆU TRƯỞNG TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN 

<|ref|>text<|/ref|><|det|>[[145, 286, 900, 350]]<|/det|>
Căn cứ Quyết định số 134/2006/QĐ-TTg, ngày 08 tháng 6 năm 2006 của Thủ tướng Chính phủ về 

In [24]:

os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name = Path(INPUT_PATH).stem

if text and text != "No text in PDF":
    with open(os.path.join(OUTPUT_DIR, f"{base_name}_markdown.md"), "w", encoding="utf-8") as f:
        f.write(markdown)
    
    if crops:
        for i, crop_img in enumerate(crops):
            crop_img.save(os.path.join(OUTPUT_DIR, f"{base_name}_figure_{i+1}.png"))
    
    print(f"Results saved to: {OUTPUT_DIR}")
    print(f"- Markdown: {base_name}_markdown.md")
    if crops:
        print(f"- {len(crops)} images extracted")
else:
    print("No text found in PDF to save")

Results saved to: /Users/jajajou1778/UIT_DOCS_AGENT/data/deepseek_ocr_results
- Markdown: 560-qd-dhcntt_5-6-2024_sua_doi_quy_dinh_dao_tao_ngoai_ngu_doi_voi_he_dai_hoc_chinh_quy_markdown.md
- 1 images extracted
